# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Publication Date: ", metadata.datePublished)
print("Identifier: ", metadata.identifier)
print("Keywords: ", metadata.keywords)
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. All references to dataset entities use their `@id` fields for consistent access.

In [ ]:
# Inspect all available record sets and their structure
print("All Record Sets (@id):")
record_sets = list(dataset.record_sets())
for rset in record_sets:
    print(f"- @id: {rset.id}, Name: {rset.name}, Description: {rset.description if hasattr(rset, 'description') else ''}")
    print("  Fields and columns:")
    for field in rset.fields:
        print(f"    • Field @id: {field.id}, name: {field.name}, type: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            for column in field.columns:
                print(f"       · Column @id: {column.id}, name: {column.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references are by their `@id` as displayed above.

In [ ]:
# Select the first record set to extract as an example
selected_record_set_id = record_sets[0].id if record_sets else None
record_set_ids = [rset.id for rset in record_sets]
dataframes = {}

print(f"Available record set @id's: {record_set_ids}")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set '{record_set_id}': shape = {dataframes[record_set_id].shape}")

if selected_record_set_id is not None:
    print(f"\nSample columns for record set '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())
    print(f"\nFirst 5 rows:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use entity `@id` strings, and adjust as needed based on the dataset structure.

In [ ]:
# Example EDA on the first available record set
import numpy as np

if selected_record_set_id is not None and not dataframes[selected_record_set_id].empty:
    df = dataframes[selected_record_set_id].copy()

    # Try to automatically find some numeric fields
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    print(f"Numeric fields in '{selected_record_set_id}':", numeric_fields)

    # Select a numeric field if available
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (75th percentile): {len(filtered_df)} rows")

        # Normalize selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical/text field
        group_fields = df.select_dtypes(include=[object, 'category']).columns.tolist()
        group_field = None
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field} (top 5):")
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, a histogram of a numeric field or a bar plot after grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and not dataframes[selected_record_set_id].empty and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {selected_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        # Plot mean value by group
        plt.figure(figsize=(8, 5))
        grp = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(15)
        sns.barplot(x=grp.values, y=grp.index, palette="mako")
        plt.title(f"Mean {numeric_field} by {group_field} (Top 15)")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded Croissant metadata and records using the `mlcroissant` library
- Explored the available record sets, their fields, and columns by their `@id`
- Extracted tabular data from the record set and performed simple EDA steps such as filtering, normalization, and grouping
- Visualized data distributions via histograms and bar plots

This process can be repeated and adapted for more detailed domain-specific analyses. Make sure to reference all dataset entities via their `@id` for programmatic reproducibility.
